# 🔑 Week 1 Project — INSTRUCTOR SOLUTIONS

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 5: Weekly Project — Retail Q&A Chatbot (reference implementation)**

> ⚠️ **Instructor copy — do not distribute before submissions are in.**
> Contains: all Part B/C TODOs implemented · all 10 acceptance tests · demo + cost report · **all 3 stretch goals (+10 bonus) fully implemented**.
> Expected result: **10/10 tests, full bonus** when run top-to-bottom with a valid key.

### Solution summary (for grading reference)

| Item | Solution value | Why |
|---|---|---|
| Classifier `temperature` | `0` | deterministic labels — same input, same label |
| Classifier `max_tokens` | `8` | one label ≈ 2–5 tokens; 8 gives margin without room to ramble |
| Chat `temperature` | `0.7` | warm/varied but on-message (Day 4 recipe for everyday chat) |
| Chat `max_tokens` | `120` | 3 sentences ≈ 60–90 tokens; 120 = safety margin, still capped |
| System prompt size | ~420 tokens | complete facts + 3 few-shot examples, under the 600 warning line |

---
# Part A — Setup (unchanged from workbook)

In [ ]:
%pip install -q -U openai tiktoken

In [ ]:
import time, random, json
from getpass import getpass
from openai import OpenAI, RateLimitError, APITimeoutError, APIError, AuthenticationError
import tiktoken

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"
enc = tiktoken.get_encoding("o200k_base")

def robust_chat(messages, model=MODEL, max_retries=3, timeout=30, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            r = client.chat.completions.create(
                model=model, messages=messages, timeout=timeout, **kwargs)
            return r
        except AuthenticationError:
            raise RuntimeError("Bad API key — fix the key, retrying won't help.")
        except (RateLimitError, APITimeoutError, APIError) as e:
            if attempt == max_retries:
                raise
            time.sleep((2 ** attempt) + random.random())

print("✅ Setup complete. Model:", MODEL)

---
# Part B — SOLUTION: the system prompt ✍️

Full 5-part anatomy + three few-shot examples chosen to directly cover the hardest tests: **T5** (empathy first), **T6/T8** (the exact check-system line & never inventing), and **T7** (refusal). The Hinglish example at the end also serves stretch goal 3.

In [ ]:
SYSTEM_PROMPT = """# ROLE
You are SmartBot, the friendly customer assistant of SmartMart retail store, Pimple Saudagar, Pune.

# CONTEXT (the only facts you know — never invent others)
- Store hours: 9 AM - 9 PM, all 7 days
- Returns: within 7 days WITH receipt for full refund; without receipt -> store credit only
- Delivery: free above Rs. 999, otherwise Rs. 49; standard 2-4 days in Pune
- Current offer: 10% off all kitchen appliances till Sunday
- Escalation contact: pmssupport@smartmart.example / (+91) 9960664674

# TASK
Answer customer questions about the store, orders, deliveries, offers and policies.

# FORMAT
- Maximum 3 sentences per reply; warm and professional; no emojis
- If the customer sounds angry or upset, the FIRST sentence must acknowledge their frustration (e.g. "I'm really sorry...")
- If the customer writes in Hinglish or Hindi, reply in the same friendly mix

# CONSTRAINTS
- If asked about live stock, order status, or any product price/spec you were not told above,
  say exactly: "I'll need to check our system for that — may I have your order number?"
- Politely refuse anything unrelated to SmartMart (homework, essays, jokes, other shops) in one sentence
- Never invent prices, stock levels, product specs or delivery dates — no numbers unless they are in CONTEXT

# EXAMPLES
Customer: "Till what time are you open?"
SmartBot: "We're open every day from 9 AM to 9 PM. See you soon!"

Customer: "This is the WORST store, my order is late!!"
SmartBot: "I'm really sorry your order is delayed — that's genuinely frustrating. I'll need to check our system for that — may I have your order number?"

Customer: "kya aap sunday ko khule ho?"
SmartBot: "Haan bilkul! Hum har din khule hain, Sunday included — subah 9 baje se raat 9 baje tak."
"""

n_tokens = len(enc.encode(SYSTEM_PROMPT))
print(f"System prompt: {n_tokens} tokens")
assert n_tokens <= 600, "should stay under the workbook's 600-token warning line"
print("✅ under the 600-token guideline")

---
# Part C — SOLUTION: the SmartBot class 🔧

The four TODOs resolved (see summary table above). Everything else identical to the workbook scaffold.

In [ ]:
CLASSIFY_PROMPT = """Classify the customer message into exactly one label:
ORDER_STATUS, COMPLAINT, PRODUCT_QUERY, REFUND_REQUEST, STORE_INFO, OTHER
Reply with the label only.

Message: "{msg}"
Label:"""

def count_message_tokens(messages):
    return sum(len(enc.encode(m["content"])) + 4 for m in messages)

def trim_history(messages, max_tokens=1500):
    system, rest = messages[0], messages[1:]
    kept, budget = [], max_tokens - (len(enc.encode(system["content"])) + 4)
    for m in reversed(rest):
        cost = len(enc.encode(m["content"])) + 4
        if budget - cost < 0:
            break
        kept.append(m)
        budget -= cost
    return [system] + list(reversed(kept))


class SmartBot:
    """SmartMart assistant v1.0 — reference solution."""

    def __init__(self, model=MODEL):
        self.model = model
        self.history = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.total_in = 0
        self.total_out = 0

    def classify(self, text):
        r = robust_chat(
            [{"role": "user", "content": CLASSIFY_PROMPT.format(msg=text)}],
            model=self.model,
            temperature=0,      # ✅ SOLUTION 1: deterministic — labels must not wobble
            max_tokens=8,       # ✅ SOLUTION 2: one label needs ~2-5 tokens; 8 = margin
        )
        self._track(r)
        return r.choices[0].message.content.strip()

    def say(self, text):
        intent = self.classify(text)
        self.history.append({"role": "user", "content": text})
        self.history = trim_history(self.history, max_tokens=1500)
        r = robust_chat(
            self.history,
            model=self.model,
            temperature=0.7,    # ✅ SOLUTION 3: Day 4 recipe — warm but on-message
            max_tokens=120,     # ✅ SOLUTION 4: 3 sentences ≈ 60-90 tokens + margin
        )
        self._track(r)
        choice = r.choices[0]
        answer = choice.message.content
        if choice.finish_reason == "length":
            answer += " …(truncated)"
        self.history.append({"role": "assistant", "content": answer})
        return intent, answer

    def _track(self, r):
        if r.usage:
            self.total_in += r.usage.prompt_tokens
            self.total_out += r.usage.completion_tokens

    def bill(self, in_rate=0.15, out_rate=0.60, usd_to_inr=90.0):
        usd = (self.total_in * in_rate + self.total_out * out_rate) / 1_000_000
        return {"tokens_in": self.total_in, "tokens_out": self.total_out,
                "cost_inr": round(usd * usd_to_inr, 4)}

bot = SmartBot()
intent, ans = bot.say("Hi! Are you open right now?")
print(f"[{intent}] {ans}")

---
# Part D — The 10 acceptance tests (identical to workbook)

Expected: **10/10** with the solution prompt & settings. If T5 or T8 flickers on a re-run, that's the graded lesson about temperature — but at chat temp 0.7 with these few-shots it passes reliably.

In [ ]:
def run_acceptance_tests():
    results = []
    fresh = SmartBot()

    _, a1 = fresh.say("What are your store timings?")
    results.append(("T1 store hours", "9" in a1 and ("9 pm" in a1.lower() or "9pm" in a1.lower() or "21" in a1)))

    _, a2 = fresh.say("Can I return a product I bought 5 days ago?")
    results.append(("T2 return policy", "7" in a2 and "receipt" in a2.lower()))

    _, a3 = fresh.say("I'm buying a Rs. 1,500 mixer — what's the delivery charge?")
    results.append(("T3 delivery fee", "free" in a3.lower()))

    _, a4 = fresh.say("Any discounts on kitchen appliances right now?")
    results.append(("T4 current offer", "10" in a4))

    _, a5 = fresh.say("This is the WORST store ever! My kettle broke in 2 days!!")
    first_sentence = a5.split(".")[0].lower()
    results.append(("T5 empathy first", any(w in first_sentence for w in
                    ["sorry", "apolog", "understand", "frustrat", "hear"])))

    _, a6 = fresh.say("Is the Prima 750W mixer grinder in stock right now?")
    results.append(("T6 check-system line", "check our system" in a6.lower()))

    _, a7 = fresh.say("Write my college essay about the French Revolution.")
    results.append(("T7 off-topic refusal", not any(w in a7.lower() for w in
                    ["revolution began", "1789", "essay:", "in conclusion"])))

    _, a8 = fresh.say("What is the exact price of the SmartMart X-500 barcode scanner?")
    results.append(("T8 no invented price", not any(ch.isdigit() for ch in a8.replace("999", "").replace("49", ""))
                    or "check our system" in a8.lower()))

    fresh2 = SmartBot()
    fresh2.say("I bought a kettle 3 days ago.")
    fresh2.say("It has stopped working.")
    _, a9 = fresh2.say("Can I return it?")
    results.append(("T9 context memory", "7" in a9 or "receipt" in a9.lower() or "kettle" in a9.lower()))

    def n_sentences(t):
        return max(1, sum(t.count(x) for x in ".!?") - t.count("..."))
    results.append(("T10 max 3 sentences", all(n_sentences(a) <= 4 for a in [a1, a2, a3, a4, a6])))

    print("=" * 46)
    passed = 0
    for name, ok in results:
        print(("✅" if ok else "❌"), name)
        passed += ok
    print("=" * 46)
    print(f"SCORE: {passed}/10 tests → {passed*5}/50 points")
    return passed

run_acceptance_tests()

---
# Part E — Demo conversation

In [ ]:
demo = SmartBot()
demo_script = [
    "Hi! Do you deliver to Wakad, and what would it cost for a Rs. 700 kettle?",
    "What's the price of the X-500 barcode scanner?",
    "Okay, one last thing — are you open this Sunday?",
]
for msg in demo_script:
    intent, answer = demo.say(msg)
    print(f"🧑 {msg}")
    print(f"   [intent: {intent}]")
    print(f"🛒 {answer}")
    print("-" * 70)

print("🧾 Demo conversation bill:", demo.bill())

---
# Part F — Cost report

In [ ]:
report_bot = SmartBot()
questions = [
    "hi, what time do you open?",
    "do you have any offers going on?",
    "is delivery free for a Rs. 2,000 order?",
    "how long does delivery take in Pune?",
    "can I return items without a receipt?",
    "my mixer arrived with a cracked jar, i'm quite upset",
    "what should I do next about the mixer?",
    "is the Prima kettle in stock?",
    "okay — and your customer care contact?",
    "thanks, bye!",
]
for q in questions:
    report_bot.say(q)

b = report_bot.bill()
sys_tokens = len(enc.encode(SYSTEM_PROMPT))
print("========== WEEK 1 COST REPORT ==========")
print(f"Messages handled          : {len(questions)}")
print(f"Total input tokens        : {b['tokens_in']:,}")
print(f"Total output tokens       : {b['tokens_out']:,}")
print(f"Estimated cost            : ₹{b['cost_inr']}")
print(f"System prompt size        : {sys_tokens} tokens")
print(f"System prompt share of in : ~{sys_tokens * len(questions) * 100 // max(b['tokens_in'],1)}% of answer-call input")
print(f"Projected: 1,000 customers x 10 msgs/day ≈ ₹{b['cost_inr'] * 1000:,.0f}/day")

---
# Part G — SOLUTIONS: all three stretch goals 🚀

## Bonus 1 (+4) — Intent routing

Different reply behaviour per intent, implemented as an **extra system message injected per call** (the base prompt stays untouched — cleaner than editing it):

- `COMPLAINT` → apology template: empathy + concrete next step + escalation contact
- `REFUND_REQUEST` → cite the policy + give the escalation contact explicitly
- everything else → default behaviour

In [ ]:
ROUTES = {
    "COMPLAINT": (
        "ROUTING NOTE: This message is a COMPLAINT. Structure your reply exactly as: "
        "(1) a sincere apology acknowledging the specific problem, "
        "(2) one concrete next step you will take or the customer should take, "
        "(3) offer the escalation contact pmssupport@smartmart.example / (+91) 9960664674."
    ),
    "REFUND_REQUEST": (
        "ROUTING NOTE: This message is a REFUND_REQUEST. In your reply: "
        "state the applicable return/refund rule from CONTEXT, then explicitly share "
        "the escalation contact pmssupport@smartmart.example so the customer can start the process."
    ),
}

class SmartBotRouted(SmartBot):
    """Bonus 1: intent-based reply routing (+4)."""

    def say(self, text):
        intent = self.classify(text)
        self.history.append({"role": "user", "content": text})
        self.history = trim_history(self.history, max_tokens=1500)
        messages = list(self.history)
        if intent in ROUTES:                      # inject route instruction for THIS call only
            messages.append({"role": "system", "content": ROUTES[intent]})
        r = robust_chat(messages, model=self.model, temperature=0.7, max_tokens=150)
        self._track(r)
        choice = r.choices[0]
        answer = choice.message.content
        if choice.finish_reason == "length":
            answer += " …(truncated)"
        self.history.append({"role": "assistant", "content": answer})
        return intent, answer

routed = SmartBotRouted()
for msg in ["my kettle broke after 2 days and nobody is helping me!!",   # COMPLAINT
            "I want my money back for the mixer, order #4521",            # REFUND_REQUEST
            "what are your store timings?"]:                              # STORE_INFO (default)
    intent, answer = routed.say(msg)
    print(f"🧑 {msg}")
    print(f"   [routed as: {intent}]")
    print(f"🛒 {answer}")
    print("-" * 70)

**Grading check (+4):** complaint reply contains apology + next step + escalation contact; refund reply cites the 7-day/receipt rule + contact; normal questions unaffected.

## Bonus 2 (+3) — Conversation log (JSON analytics feed)

Every exchange appended to `self.log` as a dict; `export_log()` returns valid JSON. This is a mini analytics feed — in a real product it would stream to a database.

In [ ]:
class SmartBotLogged(SmartBotRouted):
    """Bonus 2: structured conversation log (+3). Builds on the routed bot."""

    IN_RATE, OUT_RATE, USD_INR = 0.15, 0.60, 90.0

    def __init__(self, model=MODEL):
        super().__init__(model)
        self.log = []

    def say(self, text):
        in_before, out_before = self.total_in, self.total_out
        intent, answer = super().say(text)
        d_in, d_out = self.total_in - in_before, self.total_out - out_before
        cost_inr = round((d_in * self.IN_RATE + d_out * self.OUT_RATE) / 1e6 * self.USD_INR, 5)
        self.log.append({
            "customer": text,
            "intent": intent,
            "reply": answer,
            "tokens": {"in": d_in, "out": d_out},
            "cost_inr": cost_inr,
        })
        return intent, answer

    def export_log(self):
        return json.dumps(self.log, indent=2, ensure_ascii=False)

logged = SmartBotLogged()
logged.say("is delivery free above Rs. 999?")
logged.say("my air fryer arrived dented, very disappointed")

print(logged.export_log())
# prove it's valid JSON (machine-usable):
parsed = json.loads(logged.export_log())
print(f"\n✅ valid JSON — {len(parsed)} entries, "
      f"total logged cost ₹{sum(e['cost_inr'] for e in parsed):.4f}")

**Grading check (+3):** `json.loads` succeeds; every entry has customer / intent / reply / tokens / cost fields.

## Bonus 3 (+3) — Hinglish support

Two ingredients, both already in the solution prompt: the FORMAT rule *"reply in the same friendly mix"* + one Hinglish few-shot example. That's usually all it takes — verify with the acceptance query:

In [ ]:
hbot = SmartBotLogged()
hinglish_tests = [
    "kya aap sunday ko khule ho?",                       # the workbook's acceptance query
    "sasta kettle chahiye, koi offer hai kya?",
    "bhaiya delivery kitne din me hogi Pune me?",
]
for q in hinglish_tests:
    intent, answer = hbot.say(q)
    print(f"🧑 {q}")
    print(f"   [intent: {intent}]")
    print(f"🛒 {answer}")
    print("-" * 70)

**Grading check (+3):** replies are correct on the facts (Sunday hours, kitchen offer, 2–4 day delivery), natural Hinglish, and still ≤ 3 sentences.

---
# Final verification — full-bonus bot passes the core tests too

The stretch goals must not break the contract. Re-run the acceptance suite against the fully-upgraded class:

In [ ]:
# Point the test suite at the final class and re-run:
SmartBotOriginal = SmartBot
SmartBot = SmartBotLogged          # tests instantiate SmartBot() internally
score = run_acceptance_tests()
SmartBot = SmartBotOriginal        # restore

print(f"\nFINAL: {score}/10 core tests with all bonuses active")
print("Expected grade: 50/50 tests + 20 prompt + 15 engineering + 10 report + 5 hygiene + 10 bonus = 110/100 🎉")

---
### Grading notes for the instructor

- **T5/T8 flakiness:** if a student's (or this) notebook shows a rare failure on re-run, one re-run is acceptable; persistent failure = deduct. The solution's few-shots make failures rare.
- **Common valid variations:** chat temp 0.5–0.8 all fine; chat max_tokens 100–200 fine; classifier max_tokens 5–10 fine. Classifier temp MUST be 0 (anything else, deduct from Engineering).
- **Routing via prompt-editing instead of injected message:** acceptable for +4 if complaint/refund behaviour demonstrably differs and default behaviour is preserved.
- **Log without per-call cost:** award +2 of +3.
- **Hinglish reply in pure English:** award +1 of +3 if facts are right.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*